# New documents

In this notebook the new html documents will be created for the web navigator. There are 2 parts to this process. First we take the original documents and transform them to "Reader mode" using mozilla's readability package.The second step is to take the new documents and at to them header and footer. Header because readability only returns the body of the html so it's missing the basic structure and footer to close up the basic structure and also add the buttons to the page. After this procedure is over the old files are moved from [wp](../../data/wpcd/wp/) to [wp_old](../../data/wpcd/wp_old/) and the new files from [wp_new]() (deleted) to [wp](../../data/wpcd/wp/).

## 0. Setup

In [1]:
import os, shutil, subprocess, sys
from tqdm import tqdm
from bs4 import BeautifulSoup, Tag, NavigableString, Comment
sys.path.append('../')
from formatHTML import formatHTML
from utils import emptyFolder, getWikiNavFolder

## 1. Add caption links

In [ ]:
# Create new folder structure
for path, subdirs, files in os.walk("../../data/wpcd/wp"):
    for name in subdirs:
        old_folder = os.path.join(path, name)
        new_folder = old_folder.replace("/wp/", "/wp_new/")
        os.makedirs(new_folder, exist_ok=True)

In [ ]:
def clone(el):
    if isinstance(el, NavigableString):
        return type(el)(el)

    copy = Tag(None, el.builder, el.name, el.namespace, el.nsprefix)
    # work around bug where there is no builder set
    # https://bugs.launchpad.net/beautifulsoup/+bug/1307471
    copy.attrs = dict(el.attrs)
    for attr in ('can_be_empty_element', 'hidden'):
        setattr(copy, attr, getattr(el, attr))
    for child in el.contents:
        copy.append(clone(child))
    return copy

In [ ]:
def docCaptionsComments(pathIn, pathOut):
    # Load file
    with open(pathIn, "r", encoding='utf-8', errors='ignore') as f:
        doc = BeautifulSoup(f, "html.parser")

    # Remove comments
    comments = doc.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        comment.extract()
    
    # Change class of  captions so they appear in Readability
    thumbcaptions = doc.find_all("div", {"class": "thumbcaption"})
    for thumbcaption in thumbcaptions:
        thumbcaption["class"] = ["internal"]
    
    # Save file
    with open(pathOut, 'w', encoding='utf-8') as file:
        file.write(doc.prettify())

def decompose_nfs(pathIn, pathOut):
    with open(pathIn, "r", encoding='utf-8', errors='ignore') as f:
        doc = BeautifulSoup(f, "html.parser")

    nfs = doc.findAll("div", {"class": "NavFrame"})
    while len(nfs) > 0:
        contents = [nf.find("div", {"class": "NavContent"}) for nf in nfs]
        contents = [c.contents for c in contents if c is not None]
        navheads = [nf.find("div", {"class": "NavHead"}) for nf in nfs]
        navheads = [h.contents for h in navheads if h is not None]
        if len(contents) == 0:
            print(pathIn)
            break
        for head, content, nf in zip(navheads, contents, nfs):
            for i in content:
                j = clone(i)
                nf.insert_after(j)
            h = clone(head[0])
            nf.insert_after(h)
        for nf in nfs:
            nf.decompose()
        nfs = doc.findAll("div", {"class": "NavFrame"})
    with open(pathOut, 'w', encoding='utf-8') as file:
            file.write(doc.prettify())

for path, subdirs, files in tqdm(os.walk("../../data/wpcd/wp")):
    for name in files:
        filePath = os.path.join(path, name)
        if not filePath.endswith(".htm"):
            continue
        else:
            pathIn = filePath
            pathOut = filePath.replace("/wp/", "/wp_new/")
            docCaptionsComments(pathIn, pathOut)
            decompose_nfs(pathOut, pathOut)

## 2. Readability

In [ ]:

# Define the command as a list of strings
command = [
    'node',
    '../create_new_htm.js',
    '--sourceFile',
    'old_file',
    '--destinationFolder',
    'new_file'
]

# Copy non htm files update htm files
for path, subdirs, files in tqdm(os.walk("../../data/wpcd/wp")):
    for name in files:
        old_file = os.path.join(path, name)
        if not old_file.endswith(".htm"):
            new_file = old_file.replace("/wp/", "/wp_new/")
            shutil.copyfile(old_file, new_file)
        else:
            new_file = old_file.replace("/wp/", "/wp_new/")
            command[3] = new_file
            command[5] = new_file
            subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)        

## 2. Structure + Buttons

In [ ]:
# Define the HTML content to add at the beginning and end
header = """<html>
<head></head>
<body>
    <button id="goDown" onclick="goDown()">DOWN</button>
    <button id="goUp" onclick="goUp()">UP</button>
    <button id="get" onclick="downloadActiveElements()">GET</button>
    <button id="test" onclick="test()">TEST</button>
"""
footer = """
<script type="text/javascript" src = "../../js/buttons.js"  ></script>
</body>
</html>"""

# Copy non htm files update htm files
for path, subdirs, files in tqdm(os.walk("../../data/wpcd/wp_new")):
    for name in files:
        filePath = os.path.join(path, name)
        if not filePath.endswith(".htm"):
            continue
        else:
            with open(filePath, 'r') as file:
                original_content = file.read()
            modified_content = header + original_content + footer
            with open(filePath, 'w') as file:
                file.write(modified_content)


## 3. Delete siteSub element

In [ ]:
def del_siteSub(pathIn, pathOut):
    with open(pathIn, "r", encoding='utf-8', errors='ignore') as f:
        doc = BeautifulSoup(f, "html.parser")
    element_to_delete = doc.find('h3', id='siteSub')
    if element_to_delete:
        element_to_delete.extract()
    with open(pathOut, 'w', encoding='utf-8') as file:
            file.write(doc.prettify())

for path, subdirs, files in tqdm(os.walk("../../data/wpcd/wp_new")):
    for name in files:
        filePath = os.path.join(path, name)
        if not filePath.endswith(".htm"):
            continue
        else:
            del_siteSub(filePath, filePath)

## 4. Unwrap Tables

In [ ]:
def unwrapTable(pathIn, pathOut):
    with open(pathIn, "r", encoding='utf-8', errors='ignore') as f:
        doc = BeautifulSoup(f, "html.parser")
    bodyContent = doc.find("div", {"id":"bodyContent"})
    if bodyContent is None:
        print(pathIn)
        return
    divs = bodyContent.find_all("div")
    for div in divs:
        for child in div.children:
            if child.name == "table":
                div.insert_after(child)
    with open(pathOut, 'w', encoding='utf-8') as file:
            file.write(doc.prettify())

for path, subdirs, files in tqdm(os.walk("../../data/wpcd/wp_new")):
    for name in files:
        filePath = os.path.join(path, name)
        if not filePath.endswith(".htm"):
            continue
        else:
             unwrapTable(filePath, filePath)

## 5. Remove enlarge elements

In [ ]:
def removeEnlarge(pathIn, pathOut):
    # Load file
    with open(pathIn, "r", encoding='utf-8', errors='ignore') as f:
        doc = BeautifulSoup(f, "html.parser")

    img_elements_to_remove = doc.find_all('img', alt="Enlarge")
    for img_element in img_elements_to_remove:
        img_element.extract()
    
    # Save file
    with open(pathOut, 'w', encoding='utf-8') as file:
        file.write(doc.prettify())

for path, subdirs, files in tqdm(os.walk("../../data/wpcd/wp_new")):
    for name in files:
        filePath = os.path.join(path, name)
        if not filePath.endswith(".htm"):
            continue
        else:
             removeEnlarge(filePath, filePath)

## 6. Format function

In [ ]:
pathIn = "/Users/nearchospotamitis/EPFL/DS/MA3 (Autumn)/4. COM-412/WikiNav/data/wpcd/wp_old/u/Universe.htm"
pathOut = "/Users/nearchospotamitis/EPFL/DS/MA3 (Autumn)/4. COM-412/WikiNav/data/wpcd/wp_old/u/Universe2.htm"

activePagesPath = getWikiNavFolder() + "/activePages"

emptyFolder(activePagesPath)

## Re-organization

After this procedure is over the old files are moved from [wp](../../data/wpcd/wp/) to [wp_old](../../data/wpcd/wp_old/) and the new files from [wp_new]() (deleted) to [wp](../../data/wpcd/wp/).

1. Old files moved from [wp](../../data/wpcd/wp/) to [wp_old](../../data/wpcd/wp_old/).
2. New files moved from [wp_new]() to [wp](../../data/wpcd/wp/).
3. Folder [wp_new]() is deleted.